In [1]:
# %% [markdown]
# # Notebook 01 — Test GraFPrint : Chargement modèle & Extraction d'empreinte
#
# **Objectif** : Valider que le modèle GraFPrint se charge correctement
# et qu'une empreinte peut être extraite depuis un fichier audio.
#
# **Prérequis** :
# - `models/model_tc_29_best.pth` téléchargé depuis HuggingFace
# - `grafp/` cloné à la racine du projet
# - `pip install -r backend/requirements.txt`
#
# **Résultats attendus** :
# - Modèle chargé sans erreur
# - Empreinte de shape (D,) avec D > 0
# - Temps d'extraction < 5 secondes sur CPU

# %% [markdown]
# ## 0. Setup

# %%
import sys
import os
import time
import numpy  as np
import librosa
import torch
from pathlib import Path

# Ajout des chemins nécessaires
PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
GRAFP_ROOT   = os.path.join(PROJECT_ROOT, "grafp")
MODEL_PATH   = os.path.join(PROJECT_ROOT, "model", "model_tc_29_best.pth")

for path in [PROJECT_ROOT, GRAFP_ROOT]:
    if path not in sys.path:
        sys.path.insert(0, path)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"GRAFP_ROOT   : {GRAFP_ROOT}")
print(f"MODEL_PATH   : {MODEL_PATH}")
print(f"Torch version : {torch.__version__}")
print(f"Device        : {'CUDA' if torch.cuda.is_available() else 'CPU'}")

# %%
# Vérification que les fichiers nécessaires existent
import os.path

checks = {
    "grafp/ cloné"             : os.path.isdir(GRAFP_ROOT),
    "model_tc_29_best.pth"     : os.path.isfile(MODEL_PATH),
    "backend/core/ présent"    : os.path.isdir(os.path.join(PROJECT_ROOT, "backend", "core")),
}

all_ok = True
for label, result in checks.items():
    status = "✅" if result else "❌"
    print(f"  {status}  {label}")
    if not result:
        all_ok = False

if not all_ok:
    print("\n⚠️  Certains fichiers sont manquants.")
    print("   Modèle : https://huggingface.co/chymaera96/grafp_db/resolve/main/checkpoint.zip")
    print("   GraFPrint : git clone https://github.com/chymaera96/GraFP grafp")

# %% [markdown]
# ## 1. Chargement du FingerprintEngine

# %%
from backend.core.fingerprint_engine import FingerprintEngine

engine = FingerprintEngine()
print(f"Device sélectionné : {engine.device}")

# %%
# Chargement du modèle GraFPrint
t0 = time.time()
engine.load_grafprint_model()
elapsed = time.time() - t0

model_loaded = engine.model is not None
print(f"Modèle chargé    : {'✅ GraFPrint GNN' if model_loaded else '⚠️  Fallback Librosa'}")
print(f"Temps de chargement : {elapsed:.2f}s")

if model_loaded:
    total_params = sum(p.numel() for p in engine.model.parameters())
    print(f"Paramètres du modèle : {total_params:,}")


PROJECT_ROOT : C:\Users\Algis ADJINDA\music-antipiracy
GRAFP_ROOT   : C:\Users\Algis ADJINDA\music-antipiracy\grafp
MODEL_PATH   : C:\Users\Algis ADJINDA\music-antipiracy\model\model_tc_29_best.pth
Torch version : 2.12.0+cpu
Device        : CPU
  ✅  grafp/ cloné
  ✅  model_tc_29_best.pth
  ✅  backend/core/ présent
Device sélectionné : cpu


c:\Python311\Lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Modèle chargé    : ✅ GraFPrint GNN
Temps de chargement : 85.82s
Paramètres du modèle : 20,620,576


In [2]:
# ── Cellule de vérification rapide post-correctif ──────────────────────────
import soundfile as sf
import tempfile

def generate_test_audio(duration_sec=5.0, freq_hz=440.0, sample_rate=8000, noise_level=0.02):
    t = np.linspace(0, duration_sec, int(sample_rate * duration_sec))
    signal = np.sin(2 * np.pi * freq_hz * t).astype(np.float32)
    signal += noise_level * np.random.randn(len(signal)).astype(np.float32)
    signal = signal / np.abs(signal).max()
    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    sf.write(tmp.name, signal, sample_rate)
    return tmp.name

print("\n=== VÉRIFICATION POST-CORRECTIF ===")
mode = "GraFPrint GNN" if engine.model else "Fallback Librosa"
print(f"Mode    : {mode}")

p1 = generate_test_audio(freq_hz=440.0,  noise_level=0.01)
p2 = generate_test_audio(freq_hz=440.0,  noise_level=0.08)
p3 = generate_test_audio(freq_hz=2000.0, noise_level=0.02)

e1 = engine.extract_fingerprint(p1)
e2 = engine.extract_fingerprint(p2)
e3 = engine.extract_fingerprint(p3)

print(f"Dim     : {len(e1)}")
print(f"Finis   : {np.isfinite(e1).all()}")

s12 = engine.compare_fingerprints(e1, e2)
s13 = engine.compare_fingerprints(e1, e3)
s11 = engine.compare_fingerprints(e1, e1)

print(f"Original vs copie    : {s12:.4f}")
print(f"Original vs différent: {s13:.4f}")
print(f"Original vs lui-même : {s11:.4f}")

assert s11 >= 0.999,  "Auto-similarité incorrecte"
assert s12 > s13,     "Copie moins similaire que le contenu différent !"
assert np.isfinite(e1).all(), "NaN dans l'embedding"

print("✅ Tous les tests passent")

for p in [p1, p2, p3]:
    try: os.unlink(p)
    except: pass


=== VÉRIFICATION POST-CORRECTIF ===
Mode    : GraFPrint GNN
Dim     : 128
Finis   : True
Original vs copie    : 0.9651
Original vs différent: 0.7005
Original vs lui-même : 1.0000
✅ Tous les tests passent


In [3]:
# %% [markdown]
# ## 2. Génération d'un fichier audio synthétique de test

# %%
import soundfile as sf
import tempfile

def generate_test_audio(
    duration_sec: float = 5.0,
    freq_hz:      float = 440.0,
    sample_rate:  int   = 8000,
    noise_level:  float = 0.02
) -> str:
    """
    Génère un fichier audio WAV de test (sinusoïde + bruit).
    Retourne le chemin du fichier temporaire.
    """
    t      = np.linspace(0, duration_sec, int(sample_rate * duration_sec))
    signal = np.sin(2 * np.pi * freq_hz * t).astype(np.float32)
    signal += noise_level * np.random.randn(len(signal)).astype(np.float32)
    signal = signal / np.abs(signal).max()   # normalisation

    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    sf.write(tmp.name, signal, sample_rate)
    return tmp.name


# Génération de 3 fichiers de test
audio_original   = generate_test_audio(duration_sec=5.0, freq_hz=440.0)
audio_similar    = generate_test_audio(duration_sec=5.0, freq_hz=440.0,
                                        noise_level=0.05)  # même son + plus de bruit
audio_different  = generate_test_audio(duration_sec=5.0, freq_hz=880.0)

print(f"Audio original   : {audio_original}")
print(f"Audio similaire  : {audio_similar}  (même fréquence, bruit ×2.5)")
print(f"Audio différent  : {audio_different} (fréquence double)")

Audio original   : C:\Users\ALGISA~1\AppData\Local\Temp\tmpj7x8ehti.wav
Audio similaire  : C:\Users\ALGISA~1\AppData\Local\Temp\tmpi7qlk90l.wav  (même fréquence, bruit ×2.5)
Audio différent  : C:\Users\ALGISA~1\AppData\Local\Temp\tmp2rlae88i.wav (fréquence double)


In [4]:
# %% [markdown]
# ## 3. Extraction d'empreintes

# %%
print("Extraction des empreintes...")
print("-" * 50)

timings = {}

for label, path in [
    ("original",  audio_original),
    ("similaire", audio_similar),
    ("différent", audio_different),
]:
    t0 = time.time()
    emb = engine.extract_fingerprint(path)
    dt  = time.time() - t0

    timings[label] = dt
    print(f"  [{label:10s}]  shape={emb.shape}  "
          f"dtype={emb.dtype}  "
          f"norm={np.linalg.norm(emb):.4f}  "
          f"temps={dt:.3f}s")

print(f"\nTemps moyen d'extraction : {np.mean(list(timings.values())):.3f}s")

# %%
# Assertions de validation
emb_orig = engine.extract_fingerprint(audio_original)
emb_sim  = engine.extract_fingerprint(audio_similar)
emb_diff = engine.extract_fingerprint(audio_different)

assert emb_orig.ndim  == 1,    "L'empreinte doit être un vecteur 1D"
assert emb_orig.dtype == np.float32, "L'empreinte doit être float32"
assert len(emb_orig)  > 0,     "L'empreinte ne doit pas être vide"
assert np.isfinite(emb_orig).all(), "L'empreinte ne doit pas contenir NaN/Inf"

print(f"✅ Shape     : {emb_orig.shape}")
print(f"✅ Dtype     : {emb_orig.dtype}")
print(f"✅ Pas de NaN/Inf")
print(f"✅ Dimension : {len(emb_orig)}")


Extraction des empreintes...
--------------------------------------------------
  [original  ]  shape=(128,)  dtype=float32  norm=1.0000  temps=1.815s
  [similaire ]  shape=(128,)  dtype=float32  norm=1.0000  temps=1.908s
  [différent ]  shape=(128,)  dtype=float32  norm=1.0000  temps=1.807s

Temps moyen d'extraction : 1.844s
✅ Shape     : (128,)
✅ Dtype     : float32
✅ Pas de NaN/Inf
✅ Dimension : 128


In [5]:
# %% [markdown]
# ## 4. Comparaison de similarité

# %%
scores = {
    "original  vs similaire" : engine.compare_fingerprints(emb_orig, emb_sim),
    "original  vs différent" : engine.compare_fingerprints(emb_orig, emb_diff),
    "similaire vs différent" : engine.compare_fingerprints(emb_sim,  emb_diff),
    "original  vs lui-même"  : engine.compare_fingerprints(emb_orig, emb_orig),
}

THRESHOLD = 0.85
print(f"Seuil de détection : {THRESHOLD}\n")
print(f"{'Comparaison':35s} {'Score':8s} {'Décision':12s}")
print("-" * 58)

for label, score in scores.items():
    decision = "🔴 PIRATERIE" if score >= THRESHOLD else "🟢 LÉGAL"
    print(f"  {label:33s} {score:.4f}   {decision}")

# %%
# Validation des scores
assert 0.0 <= scores["original  vs lui-même"]  <= 1.0, "Score hors [0,1]"
assert scores["original  vs lui-même"] >= 0.95,         "Auto-similarité doit être ≈ 1.0"

print("\n✅ Tous les scores sont dans [0.0, 1.0]")
print(f"✅ Auto-similarité : {scores['original  vs lui-même']:.4f} ≥ 0.95")

Seuil de détection : 0.85

Comparaison                         Score    Décision    
----------------------------------------------------------
  original  vs similaire            0.9922   🔴 PIRATERIE
  original  vs différent            0.6661   🟢 LÉGAL
  similaire vs différent            0.6523   🟢 LÉGAL
  original  vs lui-même             1.0000   🔴 PIRATERIE

✅ Tous les scores sont dans [0.0, 1.0]
✅ Auto-similarité : 1.0000 ≥ 0.95


In [6]:
# %% [markdown]
# ## 5. Test avec bytes (upload HTTP simulé)

# %%
with open(audio_original, "rb") as f:
    audio_bytes = f.read()

t0       = time.time()
emb_bytes = engine.extract_fingerprint_from_bytes(audio_bytes)
dt        = time.time() - t0

score_vs_path = engine.compare_fingerprints(emb_orig, emb_bytes)

print(f"Extraction depuis bytes : {dt:.3f}s")
print(f"Cohérence path vs bytes : {score_vs_path:.6f}")
assert score_vs_path >= 0.99, "Les deux méthodes doivent donner des résultats quasi-identiques"
print("✅ Cohérence path / bytes validée")

Extraction depuis bytes : 1.981s
Cohérence path vs bytes : 1.000000
✅ Cohérence path / bytes validée


In [7]:
# %% [markdown]
# ## 6. Test du hash SHA-256

# %%
hash1 = FingerprintEngine.embedding_to_hash(emb_orig)
hash2 = FingerprintEngine.embedding_to_hash(emb_orig)
hash3 = FingerprintEngine.embedding_to_hash(emb_diff)

assert len(hash1) == 64,  "Hash SHA-256 doit faire 64 caractères hex"
assert hash1 == hash2,    "Le même vecteur doit donner le même hash"
assert hash1 != hash3,    "Des vecteurs différents doivent donner des hashes différents"

print(f"Hash original  : {hash1[:32]}...")
print(f"Hash différent : {hash3[:32]}...")
print("✅ Déterminisme et unicité du hash validés")

Hash original  : 9c93996131702fa36ef924be97e4068c...
Hash différent : 9690106e8d573a06031d41cddc2d658f...
✅ Déterminisme et unicité du hash validés


In [8]:
# %% [markdown]
# ## 7. Résumé

# %%
print("=" * 60)
print("  RÉSUMÉ NOTEBOOK 01 — GraFPrint Test")
print("=" * 60)
print(f"  Modèle chargé       : {'GraFPrint GNN' if engine.model else 'Fallback Librosa'}")
print(f"  Dimension empreinte : {len(emb_orig)}")
print(f"  Temps extraction    : {np.mean(list(timings.values())):.3f}s en moyenne")
print(f"  Auto-similarité     : {scores['original  vs lui-même']:.4f}")
print(f"  Seuil 0.85          : fonctionnel")
print("=" * 60)
print("  ✅ Notebook 01 validé — passer au notebook 02")

# Nettoyage fichiers temporaires
for path in [audio_original, audio_similar, audio_different]:
    os.unlink(path)

  RÉSUMÉ NOTEBOOK 01 — GraFPrint Test
  Modèle chargé       : GraFPrint GNN
  Dimension empreinte : 128
  Temps extraction    : 1.844s en moyenne
  Auto-similarité     : 1.0000
  Seuil 0.85          : fonctionnel
  ✅ Notebook 01 validé — passer au notebook 02
